In [ ]:
import torch
import torch.nn as nn

class CisSequenceEncoder(nn.Module):
    """
    Stream A: Processes the static regulatory blueprint of the gene.
    Uses a 1D-CNN architecture to act as a physical motif detector, 
    scanning for TF binding sites across a 2,000bp window.
    """
    def __init__(self, sequence_length=2000, embedding_dim=128):
        super().__init__()
        # Input shape: (Batch, 4, 2000) for one-hot encoded A, C, G, T
        self.conv_block = nn.Sequential(
            nn.Conv1d(in_channels=4, out_channels=64, kernel_size=15, stride=1, padding=7),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4), # Downsampes 2000 -> 500
            
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=9, stride=1, padding=4),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4), # Downsamples 500 -> 125
            
            nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) # Collapses sequence length to a single vector per channel
        )
        
        self.fc = nn.Sequential(
            nn.Linear(256, embedding_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def forward(self, x):
        # Transpose input to (Batch, Channels, Length) if passed as (Batch, Length, Channels)
        if x.shape[1] != 4:
            x = x.transpose(1, 2)
        x = self.conv_block(x).squeeze(-1)
        return self.fc(x)


class TransStateEncoder(nn.Module):
    """
    Stream B: Processes the dynamic global "weather report" of the cell.
    Takes the continuous shift values (Δ) of the 71 core Transcription Factors.
    """
    def __init__(self, num_tfs=71, embedding_dim=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(num_tfs, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(128, embedding_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        return self.mlp(x)


class CompositionalExpressionModel(nn.Module):
    """
    The Fusion Head: Concatenates the sequence blueprint and the cellular weather report,
    then maps their non-linear interactions to predict the final continuous ΔE response.
    """
    def __init__(self, cis_emb_dim=128, trans_emb_dim=64):
        super().__init__()
        self.cis_encoder = CisSequenceEncoder(embedding_dim=cis_emb_dim)
        self.trans_encoder = TransStateEncoder(embedding_dim=trans_emb_dim)
        
        # Combined dimension after concatenation
        fusion_dim = cis_emb_dim + trans_emb_dim
        
        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 1) # Single scalar output: Predicted ΔE for the target gene
        )

    def forward(self, x_cis, x_trans):
        # 1. Extract static structural features from the DNA sequence
        h_cis = self.cis_encoder(x_cis)
        
        # 2. Extract dynamic state features from the 71 TF shifts
        h_trans = self.trans_encoder(x_trans)
        
        # 3. Concatenate ("Compose") the two vectors
        h_fusion = torch.cat([h_cis, h_trans], dim=1)
        
        # 4. Compute physical interaction to predict final expression change
        out = self.fusion_head(h_fusion)
        return out


# --- Quick Blueprint Verification ---
if __name__ == "__main__":
    # Simulate a single batch of 32 sample points
    batch_size = 32
    
    # Fake inputs matching dataset specification
    fake_dna_sequences = torch.randn(batch_size, 2000, 4)  # 2,000bp one-hot encoded
    fake_tf_vectors = torch.randn(batch_size, 71)         # 71 core TFs activation states
    
    # Initialize and run forward pass
    model = CompositionalExpressionModel()
    predictions = model(fake_dna_sequences, fake_tf_vectors)
    
    print(f"Model successfully initiated.")
    print(f"Input Cis shape:   {fake_dna_sequences.shape}")
    print(f"Input Trans shape: {fake_tf_vectors.shape}")
    print(f"Output ΔE shape:   {predictions.shape} (Expected: [{batch_size}, 1])")